# CSCN8020 Assignment 3  
## Deep Q-Network Control of the Unitree G1 Left Elbow

**Student:** Emmanuel Ihejiamaizu  
**Student ID:** 9080005  
**Course:** CSCN8020 Reinforcement Learning  
**Repository:** `https://github.com/chooksemmanuel/CSCN8020_Assignment3`  
**Clone URL:** `https://github.com/chooksemmanuel/CSCN8020_Assignment3.git`

This notebook documents the completed student-written Deep Q-Network solution, the controlled comparison of two epsilon-decay configurations, the benchmark against the rule-based policy, and the reproducibility commands for training, evaluation, checkpoint loading, and rendering.


## 1. Project Objective

The assignment extends the validated Unitree G1 MuJoCo elbow-control workshop into a reinforcement-learning task. The agent observes the elbow state and target error, then chooses one of three discrete actions:

- decrease the elbow target,
- hold the current target,
- increase the elbow target.

A PyTorch Deep Q-Network learns this policy from replayed transitions while a separate target network stabilizes the temporal-difference targets.


In [ ]:
from pathlib import Path
import ast
import hashlib
import json
import platform
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Image, display

ROOT = Path.cwd()
print("Working directory:", ROOT)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)


## 2. Repository and Deliverable Check

This cell confirms that the main source code, selected checkpoint, experiment results, report, Brightspace PDF, and rendered video are present in the cloned repository.


In [ ]:
required_paths = [
    "src/g1_rl/g1_elbow_env.py",
    "src/g1_rl/dqn/q_network.py",
    "src/g1_rl/dqn/replay_buffer.py",
    "src/g1_rl/dqn/agent.py",
    "src/train_dqn.py",
    "src/evaluate_dqn.py",
    "src/render_dqn_policy.py",
    "models/selected_dqn.pt",
    "results/config_a/training_summary.json",
    "results/config_b/training_summary.json",
    "results/config_a/evaluation_summary.json",
    "results/config_b/evaluation_summary.json",
    "results/rule_based/evaluation_summary.json",
    "report/Unitree_G1_DQN_Technical_Report_Emmanuel_Ihejiamaizu.pdf",
    "report/CSCN8020_Assignment3_Brightspace_One_Page_Emmanuel_Ihejiamaizu.pdf",
    "demo/Unitree_G1_DQN_Demo_Final.mp4",
]

manifest = pd.DataFrame(
    {
        "path": required_paths,
        "exists": [(ROOT / p).exists() for p in required_paths],
        "size_bytes": [
            (ROOT / p).stat().st_size if (ROOT / p).exists() else None
            for p in required_paths
        ],
    }
)
display(manifest)

missing = manifest.loc[~manifest["exists"], "path"].tolist()
assert not missing, f"Missing required files: {missing}"
print("PASS: all required project files are present.")


## 3. Student-Written Implementation Structure

The solution is divided into small, testable modules:

- `G1ElbowTargetEnv` wraps the MuJoCo elbow task as a Gymnasium environment.
- `QNetwork` implements the neural action-value approximator.
- `ReplayBuffer` stores transitions and samples random mini-batches.
- `DQNAgent` handles epsilon-greedy action selection, optimization, and target-network synchronization.
- `train_dqn.py` runs training for a selected configuration.
- `evaluate_dqn.py` benchmarks DQN and rule-based policies on fixed goals.
- `render_dqn_policy.py` loads the selected checkpoint and renders the trained policy.


In [ ]:
source_files = [
    ROOT / "src/g1_rl/g1_elbow_env.py",
    ROOT / "src/g1_rl/dqn/q_network.py",
    ROOT / "src/g1_rl/dqn/replay_buffer.py",
    ROOT / "src/g1_rl/dqn/agent.py",
    ROOT / "src/train_dqn.py",
    ROOT / "src/evaluate_dqn.py",
]

rows = []
for path in source_files:
    tree = ast.parse(path.read_text(encoding="utf-8"))
    classes = [node.name for node in tree.body if isinstance(node, ast.ClassDef)]
    functions = [node.name for node in tree.body if isinstance(node, ast.FunctionDef)]
    rows.append(
        {
            "file": str(path.relative_to(ROOT)),
            "classes": ", ".join(classes) or "—",
            "top_level_functions": ", ".join(functions) or "—",
        }
    )

display(pd.DataFrame(rows))


## 4. DQN Design

### Network

The action-value network uses:

\[
4 ightarrow 64 ightarrow 64 ightarrow 3
\]

with ReLU activations between the hidden layers. The four inputs represent the compact elbow-control state, while the three outputs are the estimated Q-values for decrease, hold, and increase.

### Core learning components

- online Q-network,
- target Q-network,
- replay buffer,
- epsilon-greedy exploration,
- Huber loss,
- gradient-based optimization,
- periodic target-network synchronization,
- no bootstrapping after true terminal states,
- bootstrapping retained for time-limit truncations.


## 5. Controlled Training Comparison

Both runs use the same environment, architecture, random seed, replay strategy, optimizer settings, and training duration. The intentionally changed variable is epsilon decay:

- **Configuration A:** slower decay, `0.995`
- **Configuration B:** faster decay, `0.985`

Each configuration was trained for 1,000 episodes with seed 42.


In [ ]:
def load_json(relative_path):
    return json.loads((ROOT / relative_path).read_text(encoding="utf-8"))

training_rows = []
for label, path in [
    ("Configuration A", "results/config_a/training_summary.json"),
    ("Configuration B", "results/config_b/training_summary.json"),
]:
    data = load_json(path)
    cfg = data.get("dqn_config", {})
    training_rows.append(
        {
            "configuration": label,
            "episodes": data.get("episodes"),
            "seed": data.get("seed"),
            "epsilon_decay": cfg.get("epsilon_decay"),
            "gamma": cfg.get("gamma"),
            "learning_rate": cfg.get("learning_rate"),
            "batch_size": cfg.get("batch_size"),
            "target_sync_steps": cfg.get("target_sync_steps"),
            "final_epsilon": data.get("final_epsilon"),
            "final_20_mean_reward": data.get("final_20_mean_reward"),
            "final_50_success_rate": data.get("final_50_success_rate"),
            "training_seconds": data.get("training_seconds"),
        }
    )

training_table = pd.DataFrame(training_rows)
display(training_table)


### Recorded training outcome

Configuration A achieved a final-20 mean reward of **14.3615**, a final-50 success rate of **100%**, and trained in approximately **194.74 seconds**.

Configuration B achieved a final-20 mean reward of **14.3823**, a final-50 success rate of **100%**, and trained in approximately **144.22 seconds**.

Both configurations converged successfully. Configuration A was selected because its slower exploration decay produced a slightly stronger final benchmark profile while maintaining complete success.


In [ ]:
plot_paths = [
    "results/plots/config_a_training_reward.png",
    "results/plots/config_b_training_reward.png",
    "results/plots/epsilon_decay_comparison.png",
    "results/plots/training_loss_comparison.png",
    "results/plots/training_success_rate_comparison.png",
]

for relative_path in plot_paths:
    path = ROOT / relative_path
    if path.exists():
        print(path.name)
        display(Image(filename=str(path)))
    else:
        print("Missing plot:", relative_path)


## 6. Fixed-Goal Evaluation

The learned policies were evaluated without exploration on four benchmark targets:

\[
-0.8,\ -0.4,\ 0.4,\ 0.8
\]

Five episodes were run per target, giving 20 evaluation episodes per policy. The rule-based controller was evaluated on the same benchmark.


In [ ]:
evaluation_rows = []

for label, path in [
    ("DQN Configuration A", "results/config_a/evaluation_summary.json"),
    ("DQN Configuration B", "results/config_b/evaluation_summary.json"),
    ("Rule-based baseline", "results/rule_based/evaluation_summary.json"),
]:
    data = load_json(path)
    overall = data.get("overall", {})
    row = {"policy": label}
    row.update(overall)
    evaluation_rows.append(row)

evaluation_table = pd.DataFrame(evaluation_rows)
display(evaluation_table)


### Final benchmark results

| Policy | Success | Mean reward | Mean episode length | Mean final absolute error | Mean HOLD actions | Mean action changes |
|---|---:|---:|---:|---:|---:|---:|
| DQN Configuration A | 20/20 | 13.1796 | 19.50 | 0.0116 | 4.75 | 6.50 |
| DQN Configuration B | 20/20 | 13.1588 | 19.75 | 0.0127 | 4.50 | 6.25 |
| Rule-based baseline | 20/20 | 12.8666 | 24.00 | 0.0122 | 16.50 | 1.00 |

All policies reached every benchmark target. The selected DQN policy completed episodes faster and obtained a higher mean reward than the rule-based controller. The rule-based policy used HOLD much more often, while the DQN policy made more frequent corrective action changes.


In [ ]:
for relative_path in [
    "results/plots/configuration_comparison.csv",
    "results/plots/policy_comparison.csv",
]:
    path = ROOT / relative_path
    print(path.name)
    display(pd.read_csv(path))


In [ ]:
for relative_path in [
    "results/plots/dqn_configuration_evaluation_reward.png",
    "results/plots/evaluation_success_rate_by_goal.png",
    "results/plots/policy_episode_length_comparison.png",
]:
    path = ROOT / relative_path
    if path.exists():
        print(path.name)
        display(Image(filename=str(path)))


## 7. Selected Checkpoint Verification

`models/selected_dqn.pt` is the final selected Configuration A checkpoint used by the evaluation and rendering commands.


In [ ]:
checkpoint_path = ROOT / "models/selected_dqn.pt"

digest = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()
print("Checkpoint:", checkpoint_path)
print("Size:", checkpoint_path.stat().st_size, "bytes")
print("SHA-256:", digest)

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
print("Loaded object type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):
    print("Top-level keys:", sorted(map(str, checkpoint.keys())))
else:
    print("Checkpoint loaded successfully.")


## 8. Exact Reproduction Commands

Run all commands from the repository root after activating the virtual environment.

### Start the completed assignment notebook

```bash
jupyter notebook CSCN8020_Assignment3.ipynb
```

### Train Configuration A

```bash
PYTHONPATH=src python src/train_dqn.py --name config_a --epsilon-decay 0.995 --episodes 1000 --seed 42
```

### Train Configuration B

```bash
PYTHONPATH=src python src/train_dqn.py --name config_b --epsilon-decay 0.985 --episodes 1000 --seed 42
```

### Evaluate selected Configuration A

```bash
PYTHONPATH=src python src/evaluate_dqn.py --policy dqn --checkpoint models/selected_dqn.pt --name config_a --seed 42
```

### Evaluate Configuration B

```bash
PYTHONPATH=src python src/evaluate_dqn.py --policy dqn --checkpoint models/config_b_final.pt --name config_b --seed 42
```

### Evaluate the rule-based baseline

```bash
PYTHONPATH=src python src/evaluate_dqn.py --policy rule_based --name rule_based --seed 42
```

### Render the selected trained policy

```bash
PYTHONPATH=src python src/render_dqn_policy.py --checkpoint models/selected_dqn.pt --goals -0.8 0.8 --seed 42 --countdown 3 --step-delay 0.08
```


## 9. Interpretation

The two DQN configurations solved the elbow-targeting task reliably. The faster epsilon decay in Configuration B reduced training time, while the slower decay in Configuration A provided a marginally better evaluation reward, shorter average episode length, and lower final absolute error. The comparison demonstrates that exploration scheduling can affect the quality and efficiency of the final policy even when both runs reach a 100% success rate.

The rule-based controller remained stable and interpretable, but it was more conservative. Its larger HOLD count and longer episodes indicate that it approached targets with fewer action changes but less time efficiency.


## 10. Responsible AI Disclosure

AI tools were used for explanation, debugging support, formatting assistance, and review. The environment design, DQN implementation, experiments, evaluation decisions, interpretation, repository organization, and final submission were reviewed and validated by Emmanuel Ihejiamaizu. No external reinforcement-learning framework was used to replace the student-written DQN algorithm.


## 11. Conclusion

The assignment successfully converts the validated Unitree G1 elbow controller into a Gymnasium reinforcement-learning environment and trains a student-written DQN to control the left elbow across multiple target angles. The selected model achieved 100% benchmark success, outperformed the rule-based baseline in mean reward and episode length, and is provided with reproducible code, saved metrics, plots, a trained checkpoint, reports, and a rendered demonstration video.
